In [0]:
%run ./silver_utils

# Silver utilities

Reusable class for cleaning bronze DataFrames into silver tables.
Load it in another notebook with:

```
%run ./silver_utils
```

## Example: silver orders

Uncomment to run here, or copy into your own notebook after `%run ./silver_utils`.

In [0]:
from pyspark.sql import functions as F

(SilverTransformer.from_table("databricks_catalog.bronze_schema.orders")
    .trim("order_id", "customer_id", "product_id")
    .drop_nulls("order_id", "customer_id", "product_id", "order_date")
    .filter((F.col("quantity") > 0) & (F.col("total_amount") >= 0))
    .deduplicate("order_id")
    .cast({"total_amount": "decimal(12,2)"})
    .add_date_parts("order_date", prefix="order")
    .add_processed_at()
    .drop("_rescued_data")
    .write("databricks_catalog.silver_schema.orders"))

✓ Written to databricks_catalog.silver_schema.orders


DataFrame[order_id: string, customer_id: string, product_id: string, order_date: date, quantity: int, total_amount: decimal(12,2), order_year: int, order_month: int, processed_at: timestamp]

In [0]:
%sql
SELECT * FROM databricks_catalog.silver_schema.orders LIMIT 20;

order_id,customer_id,product_id,order_date,quantity,total_amount,order_year,order_month,processed_at
O00068,C01237,P0356,2023-10-14,4,391.64,2023,10,2026-10-09T10:02:36.931Z
O00079,C00813,P0273,2023-08-24,3,439.74,2023,8,2026-10-09T10:02:36.931Z
O00099,C00343,P0204,2024-03-26,5,8673.00,2024,3,2026-10-09T10:02:36.931Z
O00124,C01486,P0041,2023-06-08,5,2297.75,2023,6,2026-10-09T10:02:36.931Z
O00171,C00361,P0093,2023-12-02,4,2997.32,2023,12,2026-10-09T10:02:36.931Z
O00189,C00988,P0195,2024-10-31,2,1874.36,2024,10,2026-10-09T10:02:36.931Z
O00216,C01546,P0268,2023-09-21,3,1893.87,2023,9,2026-10-09T10:02:36.931Z
O00245,C00573,P0213,2023-02-18,1,1528.21,2023,2,2026-10-09T10:02:36.931Z
O00254,C00667,P0381,2023-03-25,4,4815.84,2023,3,2026-10-09T10:02:36.931Z
O00323,C01064,P0353,2024-12-08,3,4394.01,2024,12,2026-10-09T10:02:36.931Z


In [0]:
%sql
SHOW TABLES IN databricks_catalog.silver_schema;

SELECT * FROM databricks_catalog.silver_schema.orders LIMIT 20;
SELECT * FROM databricks_catalog.silver_schema.customers LIMIT 20;

customer_id,first_name,last_name,email,city,state,email_domain,full_name,processed_at
C00010,James,Myers,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,2026-10-17T11:36:31.649Z
C00045,Matthew,Lee,dkhan@hotmail.com,North Kara,OK,hotmail.com,Matthew Lee,2026-10-17T11:36:31.649Z
C00049,Mike,Harvey,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,2026-10-17T11:36:31.649Z
C00056,Tina,Cantu,cruzkristen@hotmail.com,Catherineberg,WA,hotmail.com,Tina Cantu,2026-10-17T11:36:31.649Z
C00062,Shelley,Holland,kimdennis@yahoo.com,Kaitlynburgh,MI,yahoo.com,Shelley Holland,2026-10-17T11:36:31.649Z
C00094,Hannah,Olson,rogersrandall@gonzales.org,Harrisonmouth,AK,gonzales.org,Hannah Olson,2026-10-17T11:36:31.649Z
C00104,Patrick,Meadows,gloria15@yahoo.com,New Danielle,VA,yahoo.com,Patrick Meadows,2026-10-17T11:36:31.649Z
C00105,April,Wright,christine52@kennedy.com,Crystalmouth,AL,kennedy.com,April Wright,2026-10-17T11:36:31.649Z
C00126,Amanda,King,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2026-10-17T11:36:31.649Z
C00152,David,Salazar,tara25@guzman-nelson.net,Ashleyside,ID,guzman-nelson.net,David Salazar,2026-10-17T11:36:31.649Z


In [0]:
%sql
   SELECT COUNT(*) FROM databricks_catalog.gold_schema.fact_orders;

count(1)
10000
